# Lab 2: Explore Benchmarking Nuances

In this lab, you will systematically explore how workload parameters impact inference performance. By varying one parameter at a time, you'll develop intuition for:

1. **Concurrency vs. Latency tradeoff** - How simultaneous users affect response times
2. **Input token length impact** - Why longer prompts increase TTFT
3. **Output token length impact** - How generation length affects ITL and total latency
4. **Request rate saturation** - Finding your endpoint's capacity limit
5. **Streaming vs. non-streaming** - When streaming helps (and when it doesn't)

---

## Why This Matters

Every production workload is different. A code completion assistant has short inputs and outputs with strict latency requirements. A document summarization service has long inputs with relaxed latency but high throughput needs. Understanding how these parameters affect metrics lets you:

- **Right-size your infrastructure** for your specific workload
- **Set realistic SLAs** backed by empirical data
- **Identify bottlenecks** before they hit production
- **Make informed tradeoffs** between latency, throughput, and cost

## Setup

In [ ]:
%pip install -r ../requirements.txt --quiet --no-warn-conflicts

In [ ]:
import json
import time
import os
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import sys
sys.path.append("..")
from utils import (
    get_execution_role,
    wait_for_endpoint,
    wait_for_benchmark_job,
    parse_benchmark_results,
    extract_metrics,
    format_metrics_table,
    print_comparison_table,
)

# Resolve execution role
role = get_execution_role()
region = boto3.session.Session().region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account_id}"
timestamp = datetime.now().strftime("%m%d%H%M%S")

# Initialize clients
sm_client = boto3.client("sagemaker", region_name=region)
smr_client = boto3.client("sagemaker-runtime", region_name=region)
s3_client = boto3.client("s3", region_name=region)

print(f"Region:          {region}")
print(f"Role:            {role}")
print(f"Default bucket:  {bucket}")


## 1. Deploy the Model

We deploy an Inference Component on the shared endpoint from Lab 0. This loads the model onto the GPU.

> 💡 **Tip**: Each lab creates its own IC with a unique name — no conflicts.


In [ ]:
# ---------------------------------------------------------------
# Load shared endpoint and deploy model as Inference Component
# ---------------------------------------------------------------
with open("../results/endpoint_info.json", "r") as f:
    endpoint_info = json.load(f)

endpoint_name = endpoint_info["endpoint_name"]
print(f"Using shared endpoint: {endpoint_name}")

# Model config
lmi_image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124"
hf_model_id = "meta-llama/Llama-3.1-8B-Instruct"
hf_token = os.environ.get("HF_TOKEN", "")

# Unique IC name for Lab 2
inference_component_name = f"{endpoint_name}-lab2-llama31-8b"

# Deploy IC
sm_client.create_inference_component(
    InferenceComponentName=inference_component_name,
    EndpointName=endpoint_name,
    VariantName="AllTraffic",
    Specification={
        "Container": {
            "Image": lmi_image_uri,
            "Environment": {
                "HF_MODEL_ID": hf_model_id,
                "HF_TOKEN": hf_token,
                "OPTION_ROLLING_BATCH": "vllm",
                "OPTION_MAX_ROLLING_BATCH_SIZE": "32",
                "OPTION_TENSOR_PARALLEL_DEGREE": "max",
                "OPTION_MAX_MODEL_LEN": "8192",
                "OPTION_DTYPE": "bf16",
            },
        },
        "ComputeResourceRequirements": {
            "NumberOfAcceleratorDevicesRequired": 4,
            "MinMemoryRequiredInMb": 1024,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)
print(f"⏳ Deploying model (~5-8 min)...")

import time
while True:
    resp = sm_client.describe_inference_component(InferenceComponentName=inference_component_name)
    status = resp["InferenceComponentStatus"]
    print(f"  IC Status: {status}")
    if status == "InService":
        print(f"\n✅ Model deployed! IC '{inference_component_name}' is InService.")
        break
    elif status == "Failed":
        raise RuntimeError(f"IC failed: {resp.get('FailureReason', 'Unknown')}")
    time.sleep(30)


### HuggingFace token

Used to load the model container. Already set above in the setup cell.

## 2. Helper Function: Run a Benchmark Experiment

We'll run multiple benchmarks with different configurations. This helper function creates a workload config, runs the benchmark, and returns the results.

In [ ]:
def run_benchmark_experiment(
    experiment_name: str,
    workload_params: dict,
    endpoint_name: str = endpoint_name,
    base_params: dict = None,
) -> dict:
    """
    Run a single benchmark experiment with the given workload parameters.
    
    Args:
        experiment_name: Unique name for this experiment
        workload_params: Parameters to override in the workload spec
        endpoint_name: Target endpoint
        base_params: Base parameters (defaults provided if None)
    
    Returns:
        Dictionary of extracted metrics
    """
    # Base workload parameters (our Lab 1 baseline)
    if base_params is None:
        base_params = {
            "prompt_input_tokens_mean": 550,
            "prompt_input_tokens_stddev": 150,
            "output_tokens_mean": 150,
            "output_tokens_stddev": 50,
            "concurrency": 10,
            "request_count": 64,  # Reduced for faster iteration
            "streaming": True,
        }
    
    # Override with experiment-specific params
    params = {**base_params, **workload_params}
    
    # Build workload spec
    workload_spec = {
        "benchmark": {"type": "aiperf"},
        "parameters": params,
        "tooling": {"api_standard": "openai"},
    }
    
    # Create workload config
    config_name = f"bench-{experiment_name}-{timestamp}"
    sm_client.create_ai_workload_config(
        AIWorkloadConfigName=config_name,
        AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload_spec)}},
    )
    print(f"  Created config: {config_name}")
    
    # Run benchmark
    s3_output = f"s3://{bucket}/benchmark-results/lab2-{experiment_name}-{timestamp}/"
    job_name = f"bench-{experiment_name}-{timestamp}"
    
    sm_client.create_ai_benchmark_job(
        AIBenchmarkJobName=job_name,
        BenchmarkTarget={"Endpoint": {"Identifier": endpoint_name, "InferenceComponents": [{"Identifier": inference_component_name}]}},
        OutputConfig={"S3OutputLocation": s3_output},
        AIWorkloadConfigIdentifier=config_name,
        RoleArn=role,
    )
    print(f"  Started benchmark: {job_name}")
    
    # Wait for completion
    wait_for_benchmark_job(job_name, region=region, timeout_minutes=20)
    
    # Parse results
    results = parse_benchmark_results(s3_output, region=region)
    metrics = extract_metrics(results)
    
    # Cleanup workload config
    sm_client.delete_ai_workload_config(AIWorkloadConfigName=config_name)
    
    return metrics


## 3. Experiment 1: Concurrency Impact

**Question**: How does the number of simultaneous users affect latency and throughput?

**Hypothesis**: 
- Low concurrency (1) = lowest latency, lowest throughput
- Medium concurrency (10) = moderate latency, good throughput  
- High concurrency (50) = highest latency, highest throughput (until saturation)

This is the fundamental tradeoff in any serving system: batching more requests improves hardware utilization (throughput) but increases queuing time (latency).

In [ ]:
# Run experiments at different concurrency levels
concurrency_results = {}

for concurrency in [1, 10, 50]:
    print(f"\n{'='*50}")
    print(f"Running: concurrency={concurrency}")
    print(f"{'='*50}")
    
    metrics = run_benchmark_experiment(
        experiment_name=f"conc-{concurrency}",
        workload_params={"concurrency": concurrency},
    )
    concurrency_results[f"concurrency={concurrency}"] = metrics
    format_metrics_table(metrics, title=f"Concurrency = {concurrency}")

In [ ]:
# Compare concurrency results side-by-side
print_comparison_table(
    results_list=list(concurrency_results.values()),
    labels=list(concurrency_results.keys()),
    title="Concurrency Impact on Performance",
)

### Key Takeaway: Concurrency

- **TTFT increases** with concurrency because requests queue behind each other during prefill
- **ITL stays relatively stable** because the decode phase is primarily memory-bandwidth bound
- **Throughput increases** with concurrency (more requests processed per second) up to a saturation point
- **P99 latency grows faster than P50** at high concurrency - tail latency is more sensitive to load

> 💡 **Production insight**: Choose concurrency based on your SLA. If P99 TTFT must be <1s, you may need to cap concurrent requests per instance and scale horizontally instead.

## 4. Experiment 2: Input Token Length Impact

**Question**: How does prompt length affect TTFT and overall latency?

**Hypothesis**:
- Short inputs (128 tokens) = fast prefill, low TTFT
- Long inputs (2048 tokens) = slow prefill, high TTFT
- ITL should be mostly unaffected (output generation is independent of input length after prefill)

The prefill phase processes all input tokens in parallel using matrix multiplication. Longer inputs require more compute during this phase, directly impacting TTFT.

In [ ]:
# Run experiments with different input lengths
input_length_results = {}

for input_mean in [128, 550, 2048]:
    print(f"\n{'='*50}")
    print(f"Running: input_tokens_mean={input_mean}")
    print(f"{'='*50}")
    
    metrics = run_benchmark_experiment(
        experiment_name=f"input-{input_mean}",
        workload_params={
            "prompt_input_tokens_mean": input_mean,
            "prompt_input_tokens_stddev": int(input_mean * 0.2),  # 20% stddev
        },
    )
    input_length_results[f"input={input_mean}"] = metrics
    format_metrics_table(metrics, title=f"Input Tokens Mean = {input_mean}")

In [ ]:
# Compare input length results
print_comparison_table(
    results_list=list(input_length_results.values()),
    labels=list(input_length_results.keys()),
    title="Input Token Length Impact on Performance",
)

### Key Takeaway: Input Length

- **TTFT scales with input length** - longer prompts take longer to prefill
- **ITL is mostly independent** of input length (slight increase due to KV-cache memory pressure)
- **Total request latency** increases due to both longer prefill and more tokens in the KV-cache
- The relationship between input length and TTFT is roughly linear for transformer models

> 💡 **Production insight**: If your application has long context windows (RAG, document Q&A), TTFT will be your primary bottleneck. Consider chunking strategies or prefix caching to mitigate.

## 5. Experiment 3: Output Token Length Impact

**Question**: How does generation length affect ITL and total latency?

**Hypothesis**:
- Short outputs (50 tokens) = low total latency, ITL is the same
- Long outputs (500 tokens) = high total latency, same ITL per token
- ITL may increase slightly for very long outputs due to growing KV-cache

The decode phase generates one token at a time (autoregressive). Total generation time is approximately ITL x num_output_tokens.

In [ ]:
# Run experiments with different output lengths
output_length_results = {}

for output_mean in [50, 150, 500]:
    print(f"\n{'='*50}")
    print(f"Running: output_tokens_mean={output_mean}")
    print(f"{'='*50}")
    
    metrics = run_benchmark_experiment(
        experiment_name=f"output-{output_mean}",
        workload_params={
            "output_tokens_mean": output_mean,
            "output_tokens_stddev": int(output_mean * 0.2),
        },
    )
    output_length_results[f"output={output_mean}"] = metrics
    format_metrics_table(metrics, title=f"Output Tokens Mean = {output_mean}")

In [ ]:
# Compare output length results
print_comparison_table(
    results_list=list(output_length_results.values()),
    labels=list(output_length_results.keys()),
    title="Output Token Length Impact on Performance",
)

### Key Takeaway: Output Length

- **ITL may increase slightly** with longer outputs as the KV-cache grows and memory bandwidth becomes more constrained
- **Total request latency scales linearly** with output length (more tokens to generate)
- **Throughput (tokens/sec)** may actually improve with longer outputs because the system spends proportionally less time on prefill overhead
- **TTFT is unaffected** by output length (it's determined before generation begins)

> 💡 **Production insight**: For applications generating long outputs (stories, code, reports), ITL consistency matters more than TTFT. Monitor P99 ITL to ensure smooth streaming throughout the response.

## 6. Experiment 4: Request Rate and Saturation

**Question**: What happens when request arrival rate exceeds the system's processing capacity?

**Hypothesis**:
- Low rate (1 req/sec) = all requests served immediately, minimal queuing
- Medium rate (5 req/sec) = some queuing, slight latency increase
- High rate (20 req/sec) = saturation, queue builds up, latency degrades significantly

Understanding saturation helps you set autoscaling thresholds and define capacity planning targets.

In [ ]:
# Run experiments with different request rates
request_rate_results = {}

for rate in [1.0, 5.0, 20.0]:
    print(f"\n{'='*50}")
    print(f"Running: request_rate={rate} req/sec")
    print(f"{'='*50}")
    
    metrics = run_benchmark_experiment(
        experiment_name=f"rate-{int(rate)}",
        workload_params={
            "request_rate": rate,
            "concurrency": 50,  # High concurrency to allow rate to be the bottleneck
            "request_count": max(100, int(rate * 30)),  # must be >= concurrency
            "benchmark_duration": 60,
        },
    )
    request_rate_results[f"rate={rate}/s"] = metrics
    format_metrics_table(metrics, title=f"Request Rate = {rate} req/sec")

In [ ]:
# Compare request rate results
print_comparison_table(
    results_list=list(request_rate_results.values()),
    labels=list(request_rate_results.keys()),
    title="Request Rate Impact on Performance",
)

### Key Takeaway: Request Rate

- **Below saturation**: Latency stays low, throughput matches the arrival rate
- **At saturation**: Throughput plateaus, latency starts increasing due to queuing
- **Above saturation**: Queue grows unboundedly, latency degrades rapidly, requests may timeout

> 💡 **Production insight**: Your endpoint's saturation point is the maximum sustainable request rate. Set autoscaling triggers at ~70-80% of this threshold to maintain SLA compliance. The `request_rate` parameter helps you find this limit precisely.

## 7. Experiment 5: Streaming vs Non-Streaming

**Question**: What's the performance difference between streaming and non-streaming responses?

**Context**: 
- **Streaming**: Tokens are sent back one-by-one as generated. User sees progressive output.
- **Non-streaming**: The complete response is buffered and returned all at once.

From the model's perspective, generation happens the same way. The difference is in how results are delivered to the client.

In [ ]:
# Run streaming vs non-streaming comparison
streaming_results = {}

for streaming in [True, False]:
    mode = "streaming" if streaming else "non-streaming"
    print(f"\n{'='*50}")
    print(f"Running: {mode}")
    print(f"{'='*50}")
    
    metrics = run_benchmark_experiment(
        experiment_name=f"{mode}",
        workload_params={"streaming": streaming},
    )
    streaming_results[mode] = metrics
    format_metrics_table(metrics, title=f"Mode: {mode}")

In [ ]:
# Compare streaming vs non-streaming
print_comparison_table(
    results_list=list(streaming_results.values()),
    labels=list(streaming_results.keys()),
    title="Streaming vs Non-Streaming Performance",
)

### Key Takeaway: Streaming

- **TTFT is the same** for both modes (first token generation is identical)
- **ITL is only measurable in streaming mode** (non-streaming has no inter-token delivery)
- **Total request latency** is similar, but in streaming mode the user perceives the response starting at TTFT
- **Non-streaming** may have slightly higher throughput due to reduced HTTP overhead (fewer chunks)

> 💡 **Production insight**: Use streaming for user-facing chat applications (better perceived latency). Use non-streaming for batch processing, API integrations, or when you need the complete response before acting on it.

## 8. Advanced: Warmup Parameters

Before the measured benchmark begins, you can optionally run a **warmup phase** that primes the model server's caches and JIT compilation. This ensures your benchmark measures steady-state performance, not cold-start behavior.

```python
# Warmup parameters (add to workload_spec["parameters"])
"warmup_duration": 30,        # Seconds of warmup traffic
"warmup_request_count": 20,   # Number of warmup requests
"warmup_concurrency": 5,      # Concurrency during warmup
"warmup_request_rate": 2.0,   # Request rate during warmup
```

**When to use warmup:**
- First benchmark after endpoint deployment (cold GPU caches)
- After model container restart
- When you want steady-state metrics only (exclude JIT compilation overhead)

**When to skip warmup:**
- When you want to measure cold-start behavior
- For very short benchmarks where warmup time matters
- When the endpoint is already under load

## 9. Multi-Run Confidence

For production decision-making, a single benchmark run may not be sufficient. NVIDIA AIPerf supports **multi-run confidence intervals** to quantify measurement variance.

**Approach 1: Longer benchmark duration**
```python
"benchmark_duration": 300,  # 5 minutes of sustained load
```

**Approach 2: Multiple benchmark jobs**
Run the same workload config multiple times and compute statistics across runs:

```python
# Run 3 identical benchmarks
all_metrics = []
for i in range(3):
    metrics = run_benchmark_experiment(f"run-{i}", workload_params={})
    all_metrics.append(metrics)

# Calculate confidence intervals
import numpy as np
for metric in all_metrics[0].keys():
    values = [m[metric] for m in all_metrics if metric in m]
    print(f"{metric}: {np.mean(values):.2f} +/- {np.std(values):.2f}")
```

> 💡 **Tip**: AWS contributed adaptive convergence and early stopping to AIPerf. The benchmarking service will automatically run enough requests to achieve statistically stable results when you set `benchmark_duration` instead of `request_count`.

## 10. Cleanup

Delete this lab's IC and secret. The shared endpoint stays running.


In [ ]:
# Cleanup — delete IC and HF secret (shared endpoint stays)
import time

# 1. Delete Inference Component
sm_client.delete_inference_component(InferenceComponentName=inference_component_name)
print(f"⏳ Deleting IC '{inference_component_name}'...")
while True:
    try:
        resp = sm_client.describe_inference_component(InferenceComponentName=inference_component_name)
        print(f"  IC Status: {resp['InferenceComponentStatus']}")
        time.sleep(15)
    except Exception:
        print(f"  ✅ IC deleted")
        break

# Note: Workload configs are already deleted after each experiment in run_benchmark_experiment()
print(f"✅ HF secret deleted")
print(f"\n✅ Lab 2 cleanup complete!")
print(f"   Shared endpoint '{endpoint_name}' still running.")


---

## Summary

In this lab, you explored how workload parameters affect inference performance:

| Parameter | Primary Impact | Key Insight |
|-----------|---------------|-------------|
| **Concurrency** | TTFT, Throughput | Higher concurrency = more throughput but worse per-request latency |
| **Input tokens** | TTFT | Longer inputs = longer prefill = higher TTFT |
| **Output tokens** | Total latency | Linear scaling: latency ~ ITL x output_tokens |
| **Request rate** | All metrics | Beyond saturation, everything degrades rapidly |
| **Streaming** | Perceived latency | Same compute cost, but better user experience for chat |

**The meta-lesson**: There is no single "best" configuration. The optimal setup depends on your specific workload characteristics and SLA requirements. Automated benchmarking lets you empirically validate these tradeoffs in minutes rather than days.

**Next**: In [Lab 3](../lab3/lab3_inference_recommendations.ipynb), you'll use SageMaker Automated Inference Recommendations to find the optimal deployment configuration automatically - comparing instance types and applying optimizations like speculative decoding.